In [ ]:
# SAME AS THE TREE CROWN 5M BUT 

In [ ]:
from datasets import load_dataset, Dataset

In [ ]:
ds = load_dataset(
    "restor/tcd",
    cache_dir="./data/tcd"
)
train_data = ds["train"]
test_data = ds["test"]

In [ ]:
test_data

In [ ]:
from PIL import Image

def downsample_to_gsd(example, target_gsd=10.0):
    # Extract original GSD from bounds and dimensions
    bounds = example["bounds"]          # [minx, miny, maxx, maxy]
    w, h = example["width"], example["height"]
    gsd_x = (bounds[2] - bounds[0]) / w
    gsd_y = (bounds[3] - bounds[1]) / h

    # Compute new size
    new_w = int(round(w * gsd_x / target_gsd))
    new_h = int(round(h * gsd_y / target_gsd))

    # Ensure size is at least 1x1
    new_w = max(1, new_w)
    new_h = max(1, new_h)

    image = example["image"]
    mask = example["annotation"]
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    if not isinstance(mask, Image.Image):
        mask = Image.fromarray(mask)

    image_resized = image.resize((new_w, new_h), Image.BILINEAR)
    mask_resized = mask.resize((new_w, new_h), Image.NEAREST)

    example["image"] = image_resized
    example["annotation"] = mask_resized
    example["height"], example["width"] = new_h, new_w
    return example

In [ ]:
# train_5m = ds["train"].map(downsample_to_gsd)
# test_5m  = ds["test"].map(downsample_to_gsd)
train_5m = Dataset.load_from_disk("./data/tcd_10m_sentinel_train")
test_5m  = Dataset.load_from_disk("./data/tcd_10m_sentinel_test")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample = test_5m[11]
img = np.array(sample["image"])        # (H, W, 3) RGB
mask = np.array(sample["annotation"])  # (H, W) class labels

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img)
axes[0].set_title("Image (10 m/pixel)")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray", interpolation="nearest")
axes[1].set_title("Tree crown mask (10 m)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# train_5m.save_to_disk("./data/tcd_10m_sentinel_train")
# test_5m.save_to_disk("./data/tcd_10m_sentinel_test")

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np
from PIL import Image

class TreeDataset(Dataset):
    def __init__(self, hf_dataset, target_size=(64, 64)):
        self.ds = hf_dataset
        self.target_size = target_size  # (height, width)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]

        # --- Image: force RGB ---
        img = sample["image"]
        if isinstance(img, Image.Image):
            img = img.convert("RGB")   # ensures 3 channels

        # ---(single channel) ---
        mask = sample["annotation"]
        if isinstance(mask, Image.Image):
            mask = mask.convert("L")   # grayscale
        else:
            mask = Image.fromarray(mask).convert("L")

        # Resize
        img_resized = img.resize(self.target_size, Image.BICUBIC)
        mask_resized = mask.resize(self.target_size, Image.NEAREST)

        # Convert to numpy
        img_np = np.array(img_resized).transpose(2, 0, 1).astype(np.float32) / 255.0
        mask_np = np.array(mask_resized)

        # Ensure mask is binary (0/1)
        mask_np = (mask_np > 0).astype(np.float32)

        # Convert to tensors
        img_tensor = torch.from_numpy(img_np)
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0)  # (1, H, W)

        return img_tensor, mask_tensor

In [ ]:
batch_size = 128
target_dims = (64,64)
from torch.utils.data import DataLoader
train_loader = DataLoader(
    TreeDataset(train_5m,target_dims),
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(
    TreeDataset(test_5m, target_dims),
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
for images, masks in train_loader:
    print(images.shape, masks.shape)  # Should be [16, 3, H, W] and [16, 1, H, W]
    break

In [ ]:
import matplotlib.pyplot as plt

images, masks = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    # Image
    img = images[i].permute(1, 2, 0).numpy()
    axes[0, i].imshow(img)
    axes[0, i].axis('off')
    # Mask
    mask = masks[i].squeeze().numpy()
    axes[1, i].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet50",         
    encoder_weights="imagenet",    # Pre-trained on ImageNet
    in_channels=3,                    
    classes=1,
    activation=None, 
)

model.to(device)

In [ ]:
import torch.nn as nn

dice_loss = smp.losses.DiceLoss(mode='binary')
bce_loss = nn.BCEWithLogitsLoss()
tversky_loss = smp.losses.TverskyLoss(mode='binary', alpha=0.3, beta=0.7)

def combined_loss(pred, target):
    # return dice_loss(pred, target) +0.5 * tversky_loss(pred, target)
    return tversky_loss(pred,target)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

In [ ]:
def calculate_metrics(pred, target, threshold=0.5):
    pred = torch.sigmoid(pred) > threshold
    pred = pred.float()
    target = target.float()
    
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    
    iou = (intersection + 1e-6) / (union + 1e-6)
    dice = (2 * intersection + 1e-6) / (pred.sum() + target.sum() + 1e-6)
    
    return iou.item(), dice.item()

In [ ]:
from tqdm import tqdm
import time

num_epochs = 50
best_val_iou = 0.0

# Store history for plotting
history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': []}

for epoch in range(1, num_epochs + 1):
    #   Training Phase  
    model.train()
    train_loss = 0.0
    train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} [Train]")
    
    for images, masks in train_loader_tqdm:
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward
        logits = model(images)
        loss = combined_loss(logits, masks)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_loader_tqdm.set_postfix(loss=loss.item())
    
    train_loss /= len(train_loader)
    history['train_loss'].append(train_loss)
    
    model.eval()
    val_loss = 0.0
    val_iou = 0.0
    val_dice = 0.0
    
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc=f"Epoch {epoch}/{num_epochs} [Val]"):
            images = images.to(device)
            masks = masks.to(device)
            
            logits = model(images)
            loss = combined_loss(logits, masks)
            val_loss += loss.item()
            
            # Metrics
            iou, dice = calculate_metrics(logits, masks)
            val_iou += iou
            val_dice += dice
    
    val_loss /= len(test_loader)
    val_iou /= len(test_loader)
    val_dice /= len(test_loader)
    
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_dice)
    
    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, Val IoU = {val_iou:.4f}, Val Dice = {val_dice:.4f}")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), "local_unet/best_unet_tree_seg_sentinel.pth")
        print(f"New best model saved with IoU = {val_iou:.4f}")

In [ ]:
import rasterio
from rasterio.windows import Window

In [ ]:
def plot_prediction(image_path: str, mask: np.ndarray, figsize=(15, 5)):
    alpha = 0.1
    with rasterio.open(image_path) as src:
        img = src.read([1, 2, 3])          # (3, H, W)
        img = np.moveaxis(img, 0, -1)      # (H, W, 3)
        if img.dtype != np.uint8:
            img = (img / img.max() * 255).astype(np.uint8)
    
    if mask.shape[:2] != img.shape[:2]:
        from PIL import Image
        mask_pil = Image.fromarray(mask * 255)
        mask_resized = mask_pil.resize((img.shape[1], img.shape[0]), Image.NEAREST)
        mask = (np.array(mask_resized) / 255).astype(np.uint8)
    
    overlay = np.zeros_like(img, dtype=np.uint8)
    overlay[:, :, 1] = 255  # green channel
    
    blended = img.copy().astype(np.float32)
    tree_pixels = mask == 1
    blended[tree_pixels] = (1 - alpha) * blended[tree_pixels] + alpha * overlay[tree_pixels]
    blended = blended.astype(np.uint8)
    
    # Plot
    plt.figure(figsize=figsize)
    plt.imshow(blended)
    plt.title("Satellite Image with Tree Mask Overlay")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
def predict_small_image(
    image_path: str,
    model,
    device,
    target_size: tuple = (64, 64),
    threshold: float = 0.5,
) -> np.ndarray:
    with rasterio.open(image_path) as src:
        img = src.read([1, 2, 3])          # (3, H, W)
        img = np.moveaxis(img, 0, -1)      # (H, W, 3)
        orig_h, orig_w = img.shape[:2]
        print(orig_h, orig_w)

    img_pil = Image.fromarray(img).resize(target_size, Image.BICUBIC)
    img_np = np.array(img_pil).transpose(2, 0, 1).astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_np).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = model(img_tensor)
        prob = torch.sigmoid(logits).cpu().numpy().squeeze()  # (target_h, target_w)

    # Resize probability back to original image size
    prob_pil = Image.fromarray((prob * 255).astype(np.uint8))
    prob_resized = prob_pil.resize((orig_w, orig_h), Image.NEAREST)
    mask = (np.array(prob_resized) / 255.0) > threshold
    return mask.astype(np.uint8)

In [ ]:
# NOTE: point this at a query whose input.tiff came from source_type='sentinel'
image_path = "/home/ubuntu/work/saved_data/geoai-project/geoai/storage/queries/0c1ca08a-1284-4718-9343-fa4aedd78698/input.tiff"
# image_path = "/home/ubuntu/work/saved_data/geoai-project/geoai/storage/queries/7b721135-8601-4b09-8c1f-2453b401804a/input.tiff"

target_dims = (64,64)
mask = predict_small_image(image_path, model, device, target_size=target_dims)
plot_prediction(image_path, mask)